<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Synthetic Financial Time Series

**TimeGAN: Embedder, Recovery, Supervisor, Generator, and Discriminator (PyTorch)**

&copy; Dr. Yves J. Hilpisch

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/synthts/blob/main/notebooks/04_timegan.ipynb)

## Setup

In Google Colab, select **Runtime → Run all** on a standard CPU runtime.
This cell installs `synthts` and missing dependencies automatically. Existing
installations are reused; a local checkout is preferred when available.
Before PyPI publication, installation uses the public GitHub `main` branch.

No Google Drive mount, API credentials, or uploaded files are required.
The financial examples download public end-of-day prices. Set `SYNTHTS_DATA`
to a frozen local CSV to use your own input. Sequential/TimeGAN notebooks
also accept an explicit YAML path through `SYNTHTS_CONFIG`; neither is
required for the default run. Training can take several minutes on CPU.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec('synthts') is None:
    package_source = (
        'synthts @ git+https://github.com/yhilpisch/synthts.git@main'
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        if ((directory / 'pyproject.toml').is_file()
                and (directory / 'synthts' / '__init__.py').is_file()):
            package_source = str(directory.resolve())
            break
    requirements.append(package_source)

modules = {
    'numpy': 'numpy>=1.24',
    'pandas': 'pandas>=2.0',
    'scipy': 'scipy>=1.10',
    'sklearn': 'scikit-learn>=1.3',
    'matplotlib': 'matplotlib>=3.7',
    'torch': 'torch>=2.0,<3.0',
    'yaml': 'pyyaml>=6.0',
}
for module, requirement in modules.items():
    if importlib.util.find_spec(module) is None:
        requirements.append(requirement)
if requirements:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements]
    )
    importlib.invalidate_caches()

## Imports & Settings

In [ ]:
import os
from pathlib import Path
import numpy as np
import yaml
from pylab import plt, mpl

In [ ]:
import torch
# Small networks use one CPU thread for predictable runtimes.
torch.set_num_threads(1)

In [ ]:
from synthts.data import (
    EOD_URL, load_eod, compute_log_returns, ReturnScaler, split_returns,
)
from synthts.timegan import (TimeGANEmbedder, TimeGANRecovery,
    TimeGANGenerator, TimeGANSupervisor, TimeGANDiscriminator,
    TimeGANObsDiscriminator,
    train_timegan, generate_timegan_sequences)
from synthts.evaluate import (ks_test, pass_rate, plot_histograms,
    plot_sorted, plot_pvalue_histogram, sample_windows,
    sequence_ks_pvalues, compare_acf_windows,
    plot_acf_window_comparison, plot_drawdown_window_comparison)

In [ ]:
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

CONFIG_PATH = os.environ.get('SYNTHTS_CONFIG')
if CONFIG_PATH:
    NOTEBOOK_CONFIG = yaml.safe_load(Path(CONFIG_PATH).read_text())
    CONFIG_LABEL = f'Loaded {CONFIG_PATH}'
else:
    NOTEBOOK_CONFIG = {}
    CONFIG_LABEL = 'Using illustrative defaults'

DATA_CFG = NOTEBOOK_CONFIG.get('data', {})
MODEL_CFG = NOTEBOOK_CONFIG.get('model', {})
TRAIN_CFG = NOTEBOOK_CONFIG.get('training', {})
GEN_CFG = NOTEBOOK_CONFIG.get('generation', {})

DATA_SYMBOL = DATA_CFG.get('symbol', 'GLD')
DATA_SOURCE = (os.environ.get('SYNTHTS_DATA') or DATA_CFG.get('data_path')
               or EOD_URL)
VALIDATION_FRACTION = float(DATA_CFG.get('validation_fraction', 0.2))
TEST_FRACTION = float(DATA_CFG.get('test_fraction', 0.2))
SEQ_LEN = int(MODEL_CFG.get('seq_len', 63))
INPUT_DIM = int(MODEL_CFG.get('input_dim', 1))
OUTPUT_DIM = int(MODEL_CFG.get('output_dim', 1))
HIDDEN_UNITS = int(MODEL_CFG.get('hidden_units', 64))
LATENT_DIM = int(MODEL_CFG.get('latent_dim', 24))
NOISE_DIM = int(MODEL_CFG.get('noise_dim', 8))
NUM_LAYERS = int(MODEL_CFG.get('num_layers', 1))
LATENT_ACTIVATION = MODEL_CFG.get('latent_activation', 'tanh')
OBS_CHANNELS = int(MODEL_CFG.get('obs_disc_channels', 32))
OBS_LAYERS = int(MODEL_CFG.get('obs_disc_layers', 2))
LR_SCHEDULE = TRAIN_CFG.get('lr_schedule', 'none')
CHECKPOINT_INTERVAL = int(TRAIN_CFG.get('checkpoint_interval', 200))
LOSS_WEIGHTS = {key: value for key, value in TRAIN_CFG.items()
                if key in {
                    'mean_weight', 'std_weight', 'central_quantile_weight',
                    'tail_quantile_weight', 'acf_weight', 'abs_acf_weight',
                    'squared_acf_weight', 'terminal_quantile_weight',
                    'drawdown_quantile_weight', 'obs_adv_weight'}}

EMBEDDING_EPOCHS = int(TRAIN_CFG.get('embedding_epochs', 300))
SUPERVISOR_EPOCHS = int(TRAIN_CFG.get('supervisor_epochs', 300))
JOINT_EPOCHS = int(TRAIN_CFG.get('joint_epochs', 1000))
BATCH_SIZE = int(TRAIN_CFG.get('batch_size', 128))
LR = float(TRAIN_CFG.get('lr', 0.001))
GAMMA = float(TRAIN_CFG.get('gamma', 1.0))
RECON_WEIGHT = float(TRAIN_CFG.get('recon_weight', 10.0))
SUPERVISED_WEIGHT = float(TRAIN_CFG.get('supervised_weight', 100.0))
MOMENT_WEIGHT = float(TRAIN_CFG.get('moment_weight', 100.0))
STAT_WEIGHT = float(TRAIN_CFG.get('stat_weight', 10.0))
STAT_MAX_LAG = int(TRAIN_CFG.get('stat_max_lag', 10))
TRAIN_SEED = int(TRAIN_CFG.get('seed', 100))
LOG_INTERVAL = int(TRAIN_CFG.get('log_interval', 500))

N = int(GEN_CFG.get('n_sequences', 100))
GEN_SEED = int(GEN_CFG.get('seed', 101))
WINDOW_SEED = int(GEN_CFG.get('window_seed', 100))
CONFIG_LABEL

## Data Preparation

In [ ]:
prices = load_eod(url=DATA_SOURCE, symbol=DATA_SYMBOL)
rets = compute_log_returns(prices)
scaler = ReturnScaler()
train_rets, validation_rets, test_rets = split_returns(
    rets, VALIDATION_FRACTION, TEST_FRACTION, SEQ_LEN)
rets_ = scaler.fit_transform(train_rets)
validation_scaled = scaler.transform(validation_rets)

In [ ]:
f'Returns: {len(rets)} observations'

## Model Training

TimeGAN learns a latent representation of real return windows, a recovery map back to observations, and supervised latent dynamics before adversarial joint training. The generator therefore learns temporal structure in latent space rather than only matching output-level diagnostics.

Scaling uses training returns only. Validation windows support model
selection; the final diagnostics use separate test windows. Each epoch
is one sampled minibatch update, not a pass through all observations.

In [ ]:
torch.manual_seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
embedder = TimeGANEmbedder(INPUT_DIM, HIDDEN_UNITS,
                           LATENT_DIM, NUM_LAYERS, LATENT_ACTIVATION)
recovery = TimeGANRecovery(LATENT_DIM, HIDDEN_UNITS,
                           OUTPUT_DIM, NUM_LAYERS)
generator = TimeGANGenerator(NOISE_DIM, HIDDEN_UNITS,
                             LATENT_DIM, NUM_LAYERS, LATENT_ACTIVATION)
supervisor = TimeGANSupervisor(LATENT_DIM, HIDDEN_UNITS,
                               NUM_LAYERS, LATENT_ACTIVATION)
discriminator = TimeGANDiscriminator(LATENT_DIM, HIDDEN_UNITS,
                                      NUM_LAYERS)
obs_discriminator = TimeGANObsDiscriminator(
    INPUT_DIM, OBS_CHANNELS, OBS_LAYERS)

In [ ]:
%%time
history = train_timegan(
    embedder, recovery, generator, supervisor, discriminator,
    rets_.reshape(-1, 1), seq_len=SEQ_LEN,
    embedding_epochs=EMBEDDING_EPOCHS,
    supervisor_epochs=SUPERVISOR_EPOCHS,
    joint_epochs=JOINT_EPOCHS,
    batch_size=BATCH_SIZE, noise_dim=NOISE_DIM, lr=LR,
    gamma=GAMMA, recon_weight=RECON_WEIGHT,
    supervised_weight=SUPERVISED_WEIGHT,
    moment_weight=MOMENT_WEIGHT, stat_weight=STAT_WEIGHT,
    stat_max_lag=STAT_MAX_LAG, seed=TRAIN_SEED,
    log_interval=LOG_INTERVAL, lr_schedule=LR_SCHEDULE,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    validation_data=validation_scaled, obs_discriminator=obs_discriminator,
    **LOSS_WEIGHTS
)

## Generating Synthetic Sequences

In [ ]:
seqs = generate_timegan_sequences(
    generator, supervisor, recovery, N,
    seq_len=SEQ_LEN, noise_dim=NOISE_DIM, seed=GEN_SEED
)
seqs.shape

In [ ]:
synth_windows = np.array([
    scaler.inverse_transform(seqs[i, :, 0])
    for i in range(N)
])
real_windows = sample_windows(test_rets, SEQ_LEN, N, rng_seed=WINDOW_SEED)
synth_flat = synth_windows.flatten()
real_flat = real_windows.flatten()
f'Window observations: {len(real_flat)} real, {len(synth_flat)} synthetic'

### Marginal Distribution

In [ ]:
plot_histograms(real_flat, synth_flat);

In [ ]:
plot_sorted(real_flat, synth_flat);

In [ ]:
print(f'Mean  | Real: {real_flat.mean():.6f}  Synth: {synth_flat.mean():.6f}')
print(f'Std   | Real: {real_flat.std():.6f}  Synth: {synth_flat.std():.6f}')
print(f'Min   | Real: {real_flat.min():.6f}  Synth: {synth_flat.min():.6f}')
print(f'Max   | Real: {real_flat.max():.6f}  Synth: {synth_flat.max():.6f}')

In [ ]:
ks_result = ks_test(real_flat, synth_flat)
f'Flattened window KS statistic: {ks_result["statistic"]:.4f}'

### Window KS Test

Non-rejection is a descriptive diagnostic, not evidence of equivalence.
Serial dependence and overlapping windows prevent interpreting these
p-values as calibrated independent-sample tests.

In [ ]:
pvals = sequence_ks_pvalues(real_windows, synth_windows)
f'Window KS pass rate: {pass_rate(pvals):.0%}'

In [ ]:
plot_pvalue_histogram(pvals);

### Window Autocorrelation Comparison

In [ ]:
comp = compare_acf_windows(real_windows, synth_windows,
                           max_lag=10)
f'ACF diff: {comp["acf_diff"]:.4f}, Sq-return ACF diff: {comp["acf2_diff"]:.4f}'

In [ ]:
plot_acf_window_comparison(real_windows, synth_windows,
                           max_lag=10);

### Window Cumulative Returns

In [ ]:
n_show = min(10, N)
fig, ax = plt.subplots()
days = np.arange(SEQ_LEN + 1)
for i in range(n_show):
    ax.plot(days, np.insert(np.cumsum(real_windows[i]), 0, 0),
            'r', lw=0.9, alpha=0.7, label='_nolegend_')
    ax.plot(days, np.insert(np.cumsum(synth_windows[i]), 0, 0),
            'b--', lw=0.7, alpha=0.8, label='_nolegend_')
ax.plot([], [], 'r', lw=0.9, label='real')
ax.plot([], [], 'b--', lw=0.7, label='synthetic')
ax.legend();
ax.set_title('Cumulative Log-Returns');
ax.set_xlabel('Day');
ax.set_ylabel('Cumulative Log-Return');

### Window Drawdown Comparison

In [ ]:
plot_drawdown_window_comparison(real_windows, synth_windows);

## Summary

TimeGAN separates representation learning, temporal supervision, and adversarial generation. The diagnostics compare real and synthetic windows over identical horizons.

In [ ]:
print(f'Mean  | Real: {real_flat.mean():.6f}  Synth: {synth_flat.mean():.6f}')
print(f'Std   | Real: {real_flat.std():.6f}  Synth: {synth_flat.std():.6f}')
print(f'KS stat | {ks_result["statistic"]:.4f}')
print(f'KS pass rate | {pass_rate(pvals):.0%}')
print(
    f'ACF | diff={comp["acf_diff"]:.4f} '
    f'sq-ret diff={comp["acf2_diff"]:.4f}'
)

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>